In [81]:
!apt-get -qq update
!apt-get -qq install ffmpeg
!pip -q install pysrt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [82]:
from google.colab import files

uploaded = files.upload()

Saving A1_A_good_nights_sleep.mp3 to A1_A_good_nights_sleep.mp3
Saving A1_A_good_nights_sleep.srt to A1_A_good_nights_sleep.srt


In [83]:
import csv
import zipfile
import subprocess
from pathlib import Path

import pysrt


# ==================================================
# CONFIGURACIÓN
# ==================================================

# margen en segundos
START_PADDING = 0.10
END_PADDING = 0.10

# ==================================================
# LOCALIZAR ARCHIVOS
# ==================================================

mp3_file = next(Path(".").glob("*.mp3"))
srt_file = next(Path(".").glob("*.srt"))

base_name = mp3_file.stem

output_dir = Path("output_files")
output_dir.mkdir(exist_ok=True)

tsv_file = f"{base_name}_anki.tsv"
zip_file = f"{base_name}_package.zip"

print("MP3 :", mp3_file)
print("SRT :", srt_file)
print()


# ==================================================
# FUNCIONES
# ==================================================

def to_seconds(t):
    return (
        t.hours * 3600
        + t.minutes * 60
        + t.seconds
        + t.milliseconds / 1000
    )


# ==================================================
# LEER SRT
# ==================================================

subs = pysrt.open(str(srt_file), encoding="utf-8")

print(f"Subtítulos encontrados: {len(subs)}")
print()

tsv_rows = []


# ==================================================
# CORTAR AUDIOS
# ==================================================

for index, sub in enumerate(subs, start=1):

    original_start = to_seconds(sub.start)
    original_end = to_seconds(sub.end)

    start = max(0, original_start - START_PADDING)
    end = original_end + END_PADDING

    duration = end - start

    audio_name = f"{base_name}_Line_{index:04d}.mp3"

    output_audio = output_dir / audio_name

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",

        # INPUT
        "-ss",
        f"{start:.3f}",

        "-i",
        str(mp3_file),

        "-t",
        f"{duration:.3f}",

        # AUDIO
        "-acodec",
        "libmp3lame",

        "-q:a",
        "2",

        str(output_audio)
    ]

    subprocess.run(cmd, check=True)

    text = " ".join(
        line.strip()
        for line in sub.text.splitlines()
    )

    tsv_rows.append([
        f"[sound:{audio_name}]",
        text
    ])

    print(f"✓ {audio_name}")


# ==================================================
# CREAR TSV PARA ANKI
# ==================================================

with open(
    tsv_file,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f,
        delimiter="\t"
    )

    # SIN CABECERA
    # Columna 1 = Audio
    # Columna 2 = Texto

    writer.writerows(tsv_rows)

print()
print("TSV creado:", tsv_file)


# ==================================================
# CREAR ZIP
# ==================================================

with zipfile.ZipFile(
    zip_file,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for audio in output_dir.glob("*.mp3"):

        z.write(
            audio,
            arcname=f"output_files/{audio.name}"
        )

    z.write(tsv_file)

print("ZIP creado:", zip_file)


# ==================================================
# RESUMEN
# ==================================================

print()
print("Proceso finalizado.")
print(f"Audios generados: {len(subs)}")
print(f"ZIP: {zip_file}")

MP3 : A1_A_good_nights_sleep.mp3
SRT : A1_A_good_nights_sleep.srt

Subtítulos encontrados: 25

✓ A1_A_good_nights_sleep_Line_0001.mp3
✓ A1_A_good_nights_sleep_Line_0002.mp3
✓ A1_A_good_nights_sleep_Line_0003.mp3
✓ A1_A_good_nights_sleep_Line_0004.mp3
✓ A1_A_good_nights_sleep_Line_0005.mp3
✓ A1_A_good_nights_sleep_Line_0006.mp3
✓ A1_A_good_nights_sleep_Line_0007.mp3
✓ A1_A_good_nights_sleep_Line_0008.mp3
✓ A1_A_good_nights_sleep_Line_0009.mp3
✓ A1_A_good_nights_sleep_Line_0010.mp3
✓ A1_A_good_nights_sleep_Line_0011.mp3
✓ A1_A_good_nights_sleep_Line_0012.mp3
✓ A1_A_good_nights_sleep_Line_0013.mp3
✓ A1_A_good_nights_sleep_Line_0014.mp3
✓ A1_A_good_nights_sleep_Line_0015.mp3
✓ A1_A_good_nights_sleep_Line_0016.mp3
✓ A1_A_good_nights_sleep_Line_0017.mp3
✓ A1_A_good_nights_sleep_Line_0018.mp3
✓ A1_A_good_nights_sleep_Line_0019.mp3
✓ A1_A_good_nights_sleep_Line_0020.mp3
✓ A1_A_good_nights_sleep_Line_0021.mp3
✓ A1_A_good_nights_sleep_Line_0022.mp3
✓ A1_A_good_nights_sleep_Line_0023.mp3
✓ A1_A_g

In [84]:
from google.colab import files

files.download(zip_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>